# Procrustes Residual Invoice: Geometric Distance After Optimal Alignment

Computes **Procrustes Residual** for all models from the Invoice Plot:

> **Procrustes Residual** = The literal geometric distance between the model's representations and the physical ground truth (clean embeddings) after optimal alignment.

## Interpretation

- **Clean embeddings** = reference / physical ground truth (unperturbed representation)
- **Perturbed embeddings** = model output under sequence perturbation (SNP, aa_sub, reverse, etc.)
- **Procrustes alignment** = best rotation, translation, and scaling to align perturbed → clean
- **Residual** = per-sample Frobenius distance after alignment (lower = better geometric conservation)

## Models (same as Invoice Plot)

| Category | Architectures |
|----------|---------------|
| **Transformer** | ESM-2, Nucleotide Transformer, DNABERT, DNABERT-2, AlphaGenome, OpenFold |
| **Hybrid** | HyenaDNA |
| **SSM** | Caduceus, SmallMamba |

## Prerequisites

Run after the relevant scaling/real-data experiments. Uses cached `.npy` embeddings.

---

In [ ]:
import os

# Configuration & Cache Discovery

OUTPUT_BASE = './results/procrustes_residual_invoice/'
RESULTS_DIR = OUTPUT_BASE + 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

BASE = './results'

CACHE_DIRS = {
    'ESM-2 (UniRef)':        f'{BASE}/esm2_uniref_experiment/cache',
    'ESM-2 (Synthetic)':     f'{BASE}/esm2_scaling_experiment/cache',
    'NT (RealDNA)':          f'{BASE}/nt_realdna_experiment/cache',
    'NT (Synthetic)':        f'{BASE}/nt_scaling_experiment/cache',
    'Caduceus':              f'{BASE}/caduceus_scaling_experiment/cache',
    'Caduceus (RealDNA)':    f'{BASE}/caduceus_realdna_experiment/cache',
    'HyenaDNA':              f'{BASE}/hyenadna_scaling_experiment/cache',
    'HyenaDNA (RealDNA)':    f'{BASE}/hyenadna_realdna_experiment/cache',
    'DNABERT':               f'{BASE}/dnabert_scaling_experiment/cache',
    'DNABERT (RealDNA)':     f'{BASE}/dnabert_realdna_experiment/cache',
    'DNABERT2':              f'{BASE}/dnabert2_scaling_experiment/cache',
    'DNABERT2 (RealDNA)':    f'{BASE}/dnabert2_realdna_experiment/cache',
    'OpenFold':              f'{BASE}/openfold_scaling_experiment/cache',
    'OpenFold (UniRef)':     f'{BASE}/openfold_uniref_experiment/cache',
    'Architectural Control': f'{BASE}/architectural_control_experiment/cache',
}

print(f'Results: {RESULTS_DIR}')
for name, path in CACHE_DIRS.items():
    print(f"  {name:25s} {'OK' if os.path.isdir(path) else 'NOT FOUND':>10s}  {path}")


In [ ]:
# Imports

import numpy as np
import os
import glob
from scipy.linalg import orthogonal_procrustes
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(320)
print('Ready')

In [ ]:
# Model Definitions & Cache Discovery
#
# Same models as Invoice_Plot. Each entry: (model_id, label, size_M, arch_category)
# arch_category: 'Transformer', 'HyenaDNA', 'SSM' for plotting

MODEL_DEFS = {
    'ESM-2 (UniRef)': {
        'models': [
            ('facebook/esm2_t6_8M_UR50D',    'ESM2-8M',    8, 'Transformer'),
            ('facebook/esm2_t12_35M_UR50D',   'ESM2-35M',   35, 'Transformer'),
            ('facebook/esm2_t30_150M_UR50D',  'ESM2-150M',  150, 'Transformer'),
            ('facebook/esm2_t33_650M_UR50D',  'ESM2-650M',  650, 'Transformer'),
            ('facebook/esm2_t36_3B_UR50D',    'ESM2-3B',    3000, 'Transformer'),
            ('facebook/esm2_t48_15B_UR50D',   'ESM2-15B',   15000, 'Transformer'),
        ],
        'perturbations': ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse'],
        'name_fn': lambda m: m.replace('/', '_'),
        'suffixes': ['_uniref_', '_'],
    },
    'ESM-2 (Synthetic)': {
        'models': [
            ('facebook/esm2_t6_8M_UR50D',    'ESM2-8M',    8, 'Transformer'),
            ('facebook/esm2_t12_35M_UR50D',   'ESM2-35M',   35, 'Transformer'),
            ('facebook/esm2_t30_150M_UR50D',  'ESM2-150M',  150, 'Transformer'),
            ('facebook/esm2_t33_650M_UR50D',  'ESM2-650M',  650, 'Transformer'),
            ('facebook/esm2_t36_3B_UR50D',    'ESM2-3B',    3000, 'Transformer'),
            ('facebook/esm2_t48_15B_UR50D',   'ESM2-15B',   15000, 'Transformer'),
        ],
        'perturbations': ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse'],
        'name_fn': lambda m: m.replace('/', '_'),
        'suffixes': ['_'],
    },
    'NT (RealDNA)': {
        'models': [
            ('InstaDeepAI/nucleotide-transformer-v2-50m-multi-species',  'NT-50M',   55.9, 'Transformer'),
            ('InstaDeepAI/nucleotide-transformer-v2-100m-multi-species', 'NT-100M',  113.2, 'Transformer'),
            ('InstaDeepAI/nucleotide-transformer-v2-250m-multi-species', 'NT-250M',  252.5, 'Transformer'),
            ('InstaDeepAI/nucleotide-transformer-v2-500m-multi-species', 'NT-500M',  498.5, 'Transformer'),
            ('InstaDeepAI/nucleotide-transformer-2.5b-multi-species',    'NT-2.5B',  2532.1, 'Transformer'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },
    'NT (Synthetic)': {
        'models': [
            ('InstaDeepAI/nucleotide-transformer-v2-50m-multi-species',  'NT-50M',   55.9, 'Transformer'),
            ('InstaDeepAI/nucleotide-transformer-v2-100m-multi-species', 'NT-100M',  113.2, 'Transformer'),
            ('InstaDeepAI/nucleotide-transformer-v2-250m-multi-species', 'NT-250M',  252.5, 'Transformer'),
            ('InstaDeepAI/nucleotide-transformer-v2-500m-multi-species', 'NT-500M',  498.5, 'Transformer'),
            ('InstaDeepAI/nucleotide-transformer-2.5b-multi-species',    'NT-2.5B',  2532.1, 'Transformer'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'Caduceus': {
        'models': [
            ('kuleshov-group/caduceus-ps_seqlen-1k_d_model-118_n_layer-4_lr-8e-3', 'Cad-0.5M',  0.5, 'SSM'),
            ('kuleshov-group/caduceus-ps_seqlen-1k_d_model-256_n_layer-4_lr-8e-3', 'Cad-1.9M',  1.93, 'SSM'),
            ('kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16',      'Cad-7.7M',  7.73, 'SSM'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'Caduceus (RealDNA)': {
        'models': [
            ('kuleshov-group/caduceus-ps_seqlen-1k_d_model-118_n_layer-4_lr-8e-3', 'Cad-0.5M',  0.5, 'SSM'),
            ('kuleshov-group/caduceus-ps_seqlen-1k_d_model-256_n_layer-4_lr-8e-3', 'Cad-1.9M',  1.93, 'SSM'),
            ('kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16',      'Cad-7.7M',  7.73, 'SSM'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },
    'HyenaDNA': {
        'models': [
            ('LongSafari/hyenadna-tiny-1k-seqlen-hf',         'HyenaDNA-0.5M',  0.5, 'HyenaDNA'),
            ('LongSafari/hyenadna-tiny-1k-seqlen-d256-hf',    'HyenaDNA-1.7M',  1.7, 'HyenaDNA'),
            ('LongSafari/hyenadna-small-32k-seqlen-hf',        'HyenaDNA-4.1M',  4.1, 'HyenaDNA'),
            ('LongSafari/hyenadna-medium-160k-seqlen-hf',     'HyenaDNA-14M',  14, 'HyenaDNA'),
            ('LongSafari/hyenadna-large-1m-seqlen-hf',         'HyenaDNA-55M',  55, 'HyenaDNA'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'HyenaDNA (RealDNA)': {
        'models': [
            ('LongSafari/hyenadna-tiny-1k-seqlen-hf',         'HyenaDNA-0.5M',  0.5, 'HyenaDNA'),
            ('LongSafari/hyenadna-tiny-1k-seqlen-d256-hf',    'HyenaDNA-1.7M',  1.7, 'HyenaDNA'),
            ('LongSafari/hyenadna-small-32k-seqlen-hf',       'HyenaDNA-4.1M',  4.1, 'HyenaDNA'),
            ('LongSafari/hyenadna-medium-160k-seqlen-hf',     'HyenaDNA-14M',  14, 'HyenaDNA'),
            ('LongSafari/hyenadna-large-1m-seqlen-hf',         'HyenaDNA-55M',  55, 'HyenaDNA'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },
    'DNABERT': {
        'models': [
            ('zhihan1996/DNA_bert_3', 'DNABERT-3mer', 86, 'Transformer'),
            ('zhihan1996/DNA_bert_4', 'DNABERT-4mer', 87, 'Transformer'),
            ('zhihan1996/DNA_bert_5', 'DNABERT-5mer', 88, 'Transformer'),
            ('zhihan1996/DNA_bert_6', 'DNABERT-6mer', 90, 'Transformer'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'DNABERT (RealDNA)': {
        'models': [
            ('zhihan1996/DNA_bert_3', 'DNABERT-3mer', 86, 'Transformer'),
            ('zhihan1996/DNA_bert_4', 'DNABERT-4mer', 87, 'Transformer'),
            ('zhihan1996/DNA_bert_5', 'DNABERT-5mer', 88, 'Transformer'),
            ('zhihan1996/DNA_bert_6', 'DNABERT-6mer', 90, 'Transformer'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },
    'DNABERT2': {
        'models': [
            ('zhihan1996/DNABERT-2-117M', 'DNABERT2-117M', 117, 'Transformer'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'DNABERT2 (RealDNA)': {
        'models': [
            ('zhihan1996/DNABERT-2-117M', 'DNABERT2-117M', 117, 'Transformer'),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },
        'OpenFold': {
        'perturbations': ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse'],
    },
}


def find_cache_path(cache_dir, safe_name, suffixes, tag):
    """Find cached .npy file."""
    for sfx in suffixes:
        path = os.path.join(cache_dir, f'{safe_name}{sfx}{tag}.npy')
        if os.path.exists(path):
            return path
    return None


# OpenFold: scan for openfold_L*_clean.npy, return (model_id, label, size_m, arch, cache_dir)
def discover_openfold(cache_dirs):
    found = []
    for cdir in cache_dirs:
        if not os.path.isdir(cdir):
            continue
        for f in glob.glob(os.path.join(cdir, 'openfold_*_clean.npy')):
            base = os.path.basename(f).replace('_clean.npy', '')
            length = base.replace('openfold_', '')
            found.append((base, f'OpenFold-{length}', 2100, 'Transformer', cdir))
    return found

# Architectural Control: scan for SmallMamba_*_clean.npy
def discover_smallmamba(cache_dir):
    found = []
    if not os.path.isdir(cache_dir):
        return found
    for f in glob.glob(os.path.join(cache_dir, 'SmallMamba_*_clean.npy')):
        base = os.path.basename(f).replace('_clean.npy', '')
        found.append((base, 'SmallMamba', 2.1, 'SSM'))
    return found


print('Scanning caches...\n')
available_models = []
seen = set()  # (label, size_M) to avoid duplicates from synth+real

for exp_name, exp_def in MODEL_DEFS.items():
    cache_dir = CACHE_DIRS.get(exp_name)
    if cache_dir is None or not os.path.isdir(cache_dir):
        continue

    # OpenFold: special discovery (scan both OpenFold caches)
    if 'OpenFold' in exp_name:
        of_dirs = [CACHE_DIRS.get('OpenFold'), CACHE_DIRS.get('OpenFold (UniRef)')]
        of_dirs = [d for d in of_dirs if d and os.path.isdir(d)]
        for model_id, label, size_m, arch, found_cdir in discover_openfold(of_dirs):
            key = (label, size_m)
            if key in seen:
                continue
            perts = [p for p in exp_def['perturbations']
                     if os.path.exists(os.path.join(found_cdir, f'{model_id}_{p}.npy'))]
            if len(perts) == 0:
                continue
            seen.add(key)
            available_models.append({
                'model_name': model_id, 'label': label, 'size_M': size_m, 'arch': arch,
                'experiment': exp_name, 'cache_dir': found_cdir, 'safe_name': model_id,
                'suffixes': [], 'perturbations': perts,
                'path_style': 'openfold',
            })
        continue

    for model_name, label, size_m, arch in exp_def['models']:
        safe_name = exp_def['name_fn'](model_name)
        clean_path = find_cache_path(cache_dir, safe_name, exp_def['suffixes'], 'clean')
        if clean_path is None:
            continue
        key = (label, size_m)
        if key in seen:
            continue
        perts = [p for p in exp_def['perturbations']
                 if find_cache_path(cache_dir, safe_name, exp_def['suffixes'], p)]
        if len(perts) == 0:
            continue
        seen.add(key)
        available_models.append({
            'model_name': model_name, 'label': label, 'size_M': size_m, 'arch': arch,
            'experiment': exp_name, 'cache_dir': cache_dir, 'safe_name': safe_name,
            'suffixes': exp_def['suffixes'], 'perturbations': perts,
            'path_style': 'standard',
        })
        print(f'  {label}: {len(perts)} perturbations')

# Add SmallMamba from architectural control
ac_cache = CACHE_DIRS.get('Architectural Control')
for model_id, label, size_m, arch in discover_smallmamba(ac_cache or ''):
    if ('SmallMamba', 2.1) in seen:
        continue
    perts = [p for p in ['snp_1pct', 'snp_5pct', 'snp_10pct', 'reverse_complement']
             if os.path.exists(os.path.join(ac_cache, f'{model_id}_{p}.npy'))]
    if len(perts) == 0:
        perts = [p for p in ['snp_1pct', 'snp_5pct', 'snp_10pct']
                 if os.path.exists(os.path.join(ac_cache, f'{model_id}_{p}.npy'))]
    if len(perts) > 0:
        seen.add(('SmallMamba', 2.1))
        available_models.append({
            'model_name': model_id, 'label': label, 'size_M': size_m, 'arch': arch,
            'experiment': 'Architectural Control', 'cache_dir': ac_cache, 'safe_name': model_id,
            'suffixes': [], 'perturbations': perts, 'path_style': 'standard',
        })
        print(f'  SmallMamba: {len(perts)} perturbations')

print(f'\nTotal: {len(available_models)} models')


In [ ]:
# Compute Procrustes Residual
#
# Procrustes Residual = aligned_error = geometric distance between model representation
# and physical ground truth (clean) after optimal rotation/translation/scaling.

def compute_procrustes_residual(X_clean, X_pert, n_samples=10000):
    """
    Compute Procrustes Residual: distance from clean after optimal alignment.
    Returns aligned_error (per-sample Frobenius norm).
    """
    if X_clean.shape[0] > n_samples:
        idx = np.random.choice(X_clean.shape[0], n_samples, replace=False)
        X_clean = X_clean[idx]
        X_pert = X_pert[idx]

    X_c = X_clean - X_clean.mean(axis=0)
    X_p = X_pert - X_pert.mean(axis=0)

    scale_c = np.linalg.norm(X_c, 'fro')
    scale_p = np.linalg.norm(X_p, 'fro')
    X_cn = X_c / (scale_c + 1e-12)
    X_pn = X_p / (scale_p + 1e-12)

    R, _ = orthogonal_procrustes(X_pn, X_cn)
    X_aligned = X_p @ R

    s = np.trace(X_c.T @ X_aligned) / (np.trace(X_aligned.T @ X_aligned) + 1e-12)
    X_aligned_scaled = X_aligned * s

    # Procrustes Residual = per-sample Frobenius distance after alignment
    residual = np.linalg.norm(X_c - X_aligned_scaled, 'fro') / np.sqrt(X_c.shape[0])
    return residual


print('=' * 70)
print('PROCRUSTES RESIDUAL: Geometric Distance After Optimal Alignment')
print('=' * 70)

rows = []
for minfo in available_models:
    cdir = minfo['cache_dir']
    sname = minfo['safe_name']
    sfxs = minfo['suffixes']
    path_style = minfo.get('path_style', 'standard')

    if path_style == 'openfold':
        clean_path = os.path.join(cdir, f'{sname}_clean.npy')
    else:
        clean_path = find_cache_path(cdir, sname, sfxs, 'clean')

    if not clean_path or not os.path.exists(clean_path):
        continue

    X_clean = np.load(clean_path)
    residuals = []

    for pert in minfo['perturbations']:
        if path_style == 'openfold':
            pert_path = os.path.join(cdir, f'{sname}_{pert}.npy')
        else:
            pert_path = find_cache_path(cdir, sname, sfxs, pert)
        if pert_path is None:
            continue
        X_pert = np.load(pert_path)
        res = compute_procrustes_residual(X_clean, X_pert)
        residuals.append(res)
        rows.append({'arch': minfo['arch'], 'model': minfo['label'], 'size_M': minfo['size_M'],
                    'perturbation': pert, 'procrustes_residual': res})

    if residuals:
        mean_res = np.mean(residuals)
        print(f"  {minfo['label']:20s} ({minfo['size_M']:>8.1f}M): mean residual = {mean_res:.4f}  (n={len(residuals)} perts)")

df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/procrustes_residual.csv', index=False)
print(f'\nSaved {len(df)} rows to {RESULTS_DIR}/procrustes_residual.csv')